# From Regular Expressions to <span style="font-variant:small-caps;">Nfa</span>s

This notebook shows how a given regular expression $r$ can be transformed into an equivalent finite state machine. 
It implements the theory that is outlined in section 4.4. of the 
lecture notes.

In [ ]:
#load "AutomataTypes.fsx"

In [ ]:
open AutomataTypes

The recursive `RegExp` union describes the input parse tree. NFA states are integers and the transition relation is `Map<int * Symbol, Set<int>>`. The record fields correspond directly to the five mathematical components
$$\langle Q,\Sigma,\delta,q_0,A\rangle.$$
`Eps` labels epsilon transitions, while `Input c` labels characters. The alphabet contains only input characters.

## Base automata
Each base fragment has a fresh start and a single fresh final state. `Empty` has no transition, `Epsilon` has an epsilon transition, and `Literal c` has a character transition. A character outside the supplied alphabet is rejected.

The <span style="font-variant:small-caps;">Nfa</span> `genEmptyNFA()` is defined as
$$\bigl\langle \{ q_0, q_1 \}, \Sigma, \{\}, q_0, \{ q_1 \} \bigr\rangle. $$
Note that this <span style="font-variant:small-caps;">Nfa</span> has no transitions at all.
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:

![Nfa recognizing the empty set](./aLeer.jpg)

The <span style="font-variant:small-caps;">Nfa</span> `genEpsilonNFA` is defined as
$$  \bigl\langle \{ q_0, q_1 \}, \Sigma, 
                          \bigl\{ \langle q_0, \varepsilon\rangle \mapsto \{q_1\} \bigr\}, q_0, \{ q_1 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:

![Nfa recognizing the empty string](./aEpsilon.jpg)

For a letter $c \in \Sigma$ the <span style="font-variant:small-caps;">NFA</span> `genCharNFA`$(c)$ is defined as 
$$ A(c) = 
   \bigl\langle \{ q_0, q_1 \}, \Sigma, 
   \bigl\{ \langle q_0, c \rangle \mapsto \{q_1\}\bigr\}, q_0, \{ q_1 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">NFA</span> looks as follows:

![NFA recognizing the character c](./aChar.jpg)

In [ ]:
let genEmptyNFA alphabet q0 q1 : Nfa<int> =
    { States = set [q0; q1]; Alphabet = alphabet; Transitions = Map.empty
      Start = q0; Accepting = set [q1] }

let genEpsilonNFA alphabet q0 q1 : Nfa<int> =
    { genEmptyNFA alphabet q0 q1 with Transitions = Map.ofList [ (q0, Eps), set [q1] ] }

let genCharNFA alphabet character q0 q1 : Nfa<int> =
    if not (Set.contains character alphabet) then invalidArg "character" "The character must belong to the alphabet."
    { genEmptyNFA alphabet q0 q1 with Transitions = Map.ofList [ (q0, Input character), set [q1] ] }

## Combining fragments
A Thompson fragment has exactly one accepting state. Concatenation and alternatives require disjoint states and the same alphabet. The converter at the end enforces disjointness by assigning fresh integers. Maps and sets are immutable, so composing one fragment cannot mutate another.

In [ ]:
let singleFinal (nfa: Nfa<int>) =
    if Set.count nfa.Accepting <> 1 then invalidArg "nfa" "A Thompson fragment must have exactly one final state."
    Set.minElement nfa.Accepting

let mergeFragments (first: Nfa<int>) (second: Nfa<int>) =
    if first.Alphabet <> second.Alphabet then invalidArg "second" "Fragment alphabets must agree."
    if not (Set.isEmpty (Set.intersect first.States second.States)) then
        invalidArg "second" "Thompson fragments must have disjoint state names."
    second.Transitions |> Map.fold (fun transitions key value -> Map.add key value transitions) first.Transitions

Given two <span style="font-variant:small-caps;">Nfa</span>s `f1` and `f2`, the function `catenate(f1, f2)` 
creates an <span style="font-variant:small-caps;">Nfa</span> that recognizes a string $s$ if it can be written 
in the form
$$ s = s_1s_2 $$
and $s_1$ is recognized by `f1` and $s_2$ is recognized by `f2`. 

Assume that $f_1$ and $f_2$ have the following form:
- $f_1 = \langle Q_1, \Sigma, \delta_1, q_1, \{ q_2 \}\rangle$,
- $f_2 = \langle Q_2, \Sigma, \delta_2, q_3, \{ q_4 \}\rangle$,
- $Q_1 \cap Q_2 = \{\}$.
 
Then $\texttt{catenate}(f_1, f_2)$ is defined as:
$$  \bigl\langle Q_1 \cup Q_2, \Sigma, 
   \bigl\{ \langle q_2,\varepsilon\rangle  \mapsto \{q_3\} \bigr\} 
         \cup \delta_1 \cup \delta_2, q_1, \{ q_4 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:

![Nfa recognizing the concatenation of two languages](./aConcat.jpg)

In [ ]:
let catenate (first: Nfa<int>) (second: Nfa<int>) : Nfa<int> =
    let transitions = mergeFragments first second |> Map.add (singleFinal first, Eps) (Set.singleton second.Start)
    { States = Set.union first.States second.States; Alphabet = first.Alphabet
      Transitions = transitions; Start = first.Start; Accepting = second.Accepting }

Given two <span style="font-variant:small-caps;">Nfa</span>s `f1` and `f2`, the function `disjunction(f1, f2)` 
creates an <span style="font-variant:small-caps;">Nfa</span> that recognizes a string $s$ if it is either 
is recognized by `f1` or by `f2`. 

Assume again that the states of 
$f_1$ and $f_2$ are different and that $f_1$ and $f_2$ have the following form:
- $f_1 = \langle Q_1, \Sigma, \delta_1, q_1, \{ q_3 \}\rangle$,
- $f_2 = \langle Q_2, \Sigma, \delta_2, q_2, \{ q_4 \}\rangle$,
- $Q_1 \cap Q_2 = \{\}$.

Then $\texttt{disjunction}(f_1, f_2)$ is defined as follows:
$$ \bigl\langle \{ q_0, q_5 \} \cup Q_1 \cup Q_2, \Sigma, 
                \bigl\{ \langle q_0,\varepsilon\rangle \mapsto \{q_1, q_2\},
                   \langle q_3,\varepsilon\rangle \mapsto \{q_5\}, 
                   \langle q_4,\varepsilon\rangle \mapsto \{q_5\} \bigr\} 
                   \cup \delta_1 \cup \delta_2, q_0, \{ q_5 \} \bigr\rangle
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:
![Nfa recognizing the disjunction](./aPlus.jpg)

In [ ]:
let disjunction q0 q5 (first: Nfa<int>) (second: Nfa<int>) : Nfa<int> =
    let transitions =
        mergeFragments first second
        |> Map.add (q0, Eps) (set [first.Start; second.Start])
        |> Map.add (singleFinal first, Eps) (Set.singleton q5)
        |> Map.add (singleFinal second, Eps) (Set.singleton q5)
    { States = Set.unionMany [set [q0; q5]; first.States; second.States]
      Alphabet = first.Alphabet; Transitions = transitions; Start = q0; Accepting = set [q5] }

Given an <span style="font-variant:small-caps;">Nfa</span> `f`, the function `kleene(f)` 
creates an <span style="font-variant:small-caps;">Nfa</span> that recognizes a string $s$ if it can be written as
$$ s = s_1 s_2 \cdots s_n $$
and all $s_i$ are recognized by `f`.  Note that $n$ might be $0$. 

If `f` is defined as
$$ f = \langle Q, \Sigma, \delta, q_1, \{ q_2 \} \rangle,
$$
then  `kleene(f)` is defined as follows:
$$ \bigl\langle \{ q_0, q_3 \} \cup Q, \Sigma, 
                \bigl\{ \langle q_0,\varepsilon\rangle \mapsto \{q_1, q_3\},  
                        \langle q_2,\varepsilon\rangle \mapsto \{q_1, q_3\} \bigr\} 
                \cup \delta, q_0, \{ q_3 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:
![Nfa recognizing the Kleene star](./aStar.jpg)

In [ ]:
let kleene q0 q3 (fragment: Nfa<int>) : Nfa<int> =
    let transitions =
        fragment.Transitions
        |> Map.add (q0, Eps) (set [fragment.Start; q3])
        |> Map.add (singleFinal fragment, Eps) (set [fragment.Start; q3])
    { fragment with States = Set.union (set [q0; q3]) fragment.States
                    Transitions = transitions; Start = q0; Accepting = set [q3] }

## Recursive conversion
`regexp2nfa alphabet expression` performs structural recursion on the union. A local counter implements `getNewState`, replacing the Python class and its later method replacements. The counter is confined to one conversion; rerunning a notebook cell starts with the same state names. Each recursive case calls the construction corresponding to its mathematical operator.

In [ ]:
let regexp2nfa (alphabet: Set<char>) (expression: RegExp) : Nfa<int> =
    // State allocation is local to one conversion. No previous run affects its result.
    let mutable stateCount = 0
    let getNewState () =
        stateCount <- stateCount + 1
        stateCount
    let rec toNFA expression =
        match expression with
        | Empty -> genEmptyNFA alphabet (getNewState ()) (getNewState ())
        | Epsilon -> genEpsilonNFA alphabet (getNewState ()) (getNewState ())
        | Literal c -> genCharNFA alphabet c (getNewState ()) (getNewState ())
        | Concat (left, right) ->
            let first = toNFA left
            let second = toNFA right
            catenate first second
        | Alt (left, right) ->
            let first = toNFA left
            let second = toNFA right
            disjunction (getNewState ()) (getNewState ()) first second
        | Star expression ->
            let fragment = toNFA expression
            kleene (getNewState ()) (getNewState ()) fragment
    toNFA expression

The notebook `04-Test-Regexp-2-NFA`can be used to test the functions implemented in this notebook.